# Lesson 11 — MLOps Infrastructure & Orchestration

**Course:** [ML in Practice](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/11-mlops-infrastructure-and-orchestration)

Three self-contained NumPy + matplotlib demos for the three load-bearing ideas in the lesson:

1. **Toy DAG executor** — build a tiny scheduler that runs tasks in topological order, retries on simulated transient failures with exponential backoff, and skips downstream tasks when a parent permanently fails.
2. **Queueing theory** — plot expected wait time $W_q = \rho / (\mu (1 - \rho))$ for an M/M/1 queue and watch it blow up as utilisation $\rho = \lambda/\mu \to 1$.
3. **Fair-share GPU scheduler** — simulate sharing $N$ GPUs across $K$ jobs from $T$ teams with a round-robin fair-share policy; measure makespan and per-team latency.

Self-contained: NumPy + matplotlib only. No torch, no sklearn, no scipy, no network, no API keys.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File -> Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque

# Dark style matching the site theme.
plt.style.use('dark_background')
plt.rcParams.update({
    'axes.edgecolor': '#475569',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.titlecolor': '#e2e8f0',
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'grid.color': '#2e3347',
    'savefig.facecolor': '#0f1117',
})
BRAND = '#6366f1'
TEAL = '#14b8a6'
ROSE = '#f43f5e'
YELLOW = '#eab308'

rng = np.random.default_rng(0)

## 1. A toy DAG executor with retries

We build the smallest scheduler that demonstrates the three properties a real orchestrator gives you:

- **Topological ordering** — a task runs only after all its parents succeed.
- **Retries with exponential backoff** — transient failures don't kill the run; the executor retries each task up to `max_retries` times with delays `base * 2**attempt`.
- **Failure propagation** — if a task permanently fails, its downstream tasks are skipped (not retried, not silently treated as success).

We model task failures as Bernoulli with a per-task probability so the same code can simulate flaky vs reliable tasks. The 'run' here is just `time.sleep`-free book-keeping — enough to see the order, the retries, and the skip behaviour.

In [ ]:
def topo_sort(tasks, edges):
    """Kahn's algorithm. `tasks` is a list of names, `edges` a list of (parent, child)."""
    incoming = {t: 0 for t in tasks}
    children = defaultdict(list)
    for parent, child in edges:
        incoming[child] += 1
        children[parent].append(child)
    ready = deque([t for t, n in incoming.items() if n == 0])
    order = []
    while ready:
        t = ready.popleft()
        order.append(t)
        for c in children[t]:
            incoming[c] -= 1
            if incoming[c] == 0:
                ready.append(c)
    if len(order) != len(tasks):
        raise ValueError('graph has a cycle')
    return order, children


def run_dag(tasks, edges, fail_prob, max_retries=3, seed=0):
    """Run a DAG with per-task Bernoulli failures and exponential-backoff retries.

    Returns a per-task dict with `status` (success/failed/skipped) and `attempts`.
    """
    rng_local = np.random.default_rng(seed)
    order, children = topo_sort(tasks, edges)
    # Quick parent lookup.
    parents = defaultdict(list)
    for parent, child in edges:
        parents[child].append(parent)

    log = {}
    for t in order:
        # Skip if any parent failed.
        if any(log.get(p, {}).get('status') != 'success' for p in parents[t]):
            log[t] = {'status': 'skipped', 'attempts': 0}
            continue
        # Try up to max_retries + 1 attempts.
        for attempt in range(max_retries + 1):
            failed = rng_local.random() < fail_prob[t]
            if not failed:
                log[t] = {'status': 'success', 'attempts': attempt + 1}
                break
        else:
            log[t] = {'status': 'failed', 'attempts': max_retries + 1}
    return order, log

# Example DAG: ingest -> transform -> [train, validate] -> register
tasks = ['ingest', 'transform', 'train', 'validate', 'register']
edges = [('ingest', 'transform'), ('transform', 'train'),
         ('transform', 'validate'), ('train', 'register'),
         ('validate', 'register')]
# `transform` is flaky (60% failure per attempt); everything else mostly works.
fail_prob = {'ingest': 0.05, 'transform': 0.6, 'train': 0.1, 'validate': 0.1, 'register': 0.05}

order, log = run_dag(tasks, edges, fail_prob, max_retries=3, seed=7)
print('Topological order:', ' -> '.join(order))
print()
for t in order:
    print(f'  {t:10s}  status={log[t]["status"]:7s}  attempts={log[t]["attempts"]}')

Run the executor 1000 times to estimate how often the whole pipeline succeeds. Without retries the flaky `transform` step would kill ~60% of runs and the success rate would be roughly $0.4 \cdot 0.9 \cdot 0.9 \cdot 0.95 \approx 31\%$. With three retries on each step the success rate jumps dramatically — four chances to clear a 60%-failure step gives $1 - 0.6^4 = 87\%$ for that step alone.

In [ ]:
def end_to_end_success_rate(tasks, edges, fail_prob, max_retries, n_runs=1000):
    successes = 0
    for seed in range(n_runs):
        _, log = run_dag(tasks, edges, fail_prob, max_retries=max_retries, seed=seed)
        if log['register']['status'] == 'success':
            successes += 1
    return successes / n_runs

retries_to_try = [0, 1, 2, 3, 5]
rates = [end_to_end_success_rate(tasks, edges, fail_prob, r) for r in retries_to_try]
for r, rate in zip(retries_to_try, rates):
    print(f'max_retries={r}: end-to-end success rate = {rate:.2%}')

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(retries_to_try, [r * 100 for r in rates], 'o-', color=BRAND, lw=2, markersize=8)
ax.set_xlabel('max retries per task')
ax.set_ylabel('end-to-end success rate (%)')
ax.set_title('Retries turn a flaky pipeline into a reliable one')
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Three retries take a pipeline that succeeds ~30% of the time into one that succeeds >85% of the time, without touching the (still flaky) underlying `transform` step. That is the operational value of an orchestrator over a bare cron job: it converts transient failures into invisible retries instead of paging humans.

Two caveats that the toy doesn't capture: real retries must be **idempotent** (otherwise the retried attempt corrupts state), and you should distinguish *transient* failures (retry with backoff) from *deterministic* ones (a malformed input shouldn't be retried 5 times — it should fail fast).

## 2. Queueing theory: why 95% utilisation feels broken

For an M/M/1 queue (Poisson arrivals, exponential service times, one server), the expected wait time in the queue before a job starts is

$$W_q = \frac{\rho}{\mu (1 - \rho)}$$

where $\rho = \lambda / \mu$ is the utilisation. As $\rho \to 1$ the queue wait time goes to infinity — a 'fully utilised' cluster is a cluster with unbounded wait variance. We plot $W_q$ in units of *service times* ($1/\mu$) so the curve doesn't depend on absolute throughput.

Cross-check the formula by Monte Carlo simulation: simulate 200 000 jobs arriving according to a Poisson process, FIFO-served by one worker, and measure the empirical mean wait. The simulated mean should track the closed-form curve.

In [ ]:
def simulate_mm1(rho, mu=1.0, n_jobs=200_000, seed=0):
    """M/M/1 FIFO single-server simulation. Returns empirical mean wait time in queue."""
    lam = rho * mu
    rng_local = np.random.default_rng(seed)
    inter_arrivals = rng_local.exponential(1.0 / lam, size=n_jobs)
    service_times = rng_local.exponential(1.0 / mu, size=n_jobs)
    arrivals = np.cumsum(inter_arrivals)
    start_times = np.empty(n_jobs)
    finish_times = np.empty(n_jobs)
    start_times[0] = arrivals[0]
    finish_times[0] = start_times[0] + service_times[0]
    for i in range(1, n_jobs):
        start_times[i] = max(arrivals[i], finish_times[i - 1])
        finish_times[i] = start_times[i] + service_times[i]
    wait_in_queue = start_times - arrivals
    # Drop the first few thousand to let the queue reach steady state.
    return float(np.mean(wait_in_queue[20_000:]))

rhos = np.array([0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95])
mu = 1.0
theory = rhos / (mu * (1 - rhos))
empirical = np.array([simulate_mm1(r, mu=mu, n_jobs=200_000, seed=int(100 * r)) for r in rhos])

for r, t, e in zip(rhos, theory, empirical):
    print(f'rho={r:.2f}:  theory Wq = {t:6.2f}  service-times,  empirical = {e:6.2f}')

rho_curve = np.linspace(0.01, 0.99, 200)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(rho_curve, rho_curve / (mu * (1 - rho_curve)), color=BRAND, lw=2,
        label=r'theory: $W_q = \rho / (\mu(1 - \rho))$')
ax.plot(rhos, empirical, 'o', color=ROSE, markersize=8, label='M/M/1 simulation (mean wait)')
ax.axvline(0.8, color=YELLOW, linestyle='--', lw=1, alpha=0.7, label='operational target ~0.8')
ax.set_xlabel(r'utilisation $\rho = \lambda / \mu$')
ax.set_ylabel(r'expected wait time $W_q$  (units of $1/\mu$)')
ax.set_title('M/M/1 wait time blows up as utilisation approaches 1')
ax.set_xlim(0, 1)
ax.set_ylim(0, 25)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

At $\rho = 0.5$ the average wait is one service-time. At $\rho = 0.9$ it's nine. At $\rho = 0.95$ it's nineteen. The empirical points should line up almost exactly with the closed-form curve. This is the formula that gives 'keep clusters at 70–80% utilisation' its mathematical force: the cost of the last 15% of utilisation is most of the perceived wait time.

Real ML clusters have multiple servers (M/M/c) and heavier-tailed service distributions, which shift the constants but not the shape. The principle is the same: an unbounded queue under near-saturation is the substrate of every 'my training job hasn't started yet' complaint.

## 3. Fair-share GPU scheduling

Three teams submit GPU jobs to a shared pool. Without fair-share, the team that submits first hoards the cluster; with fair-share, each team gets a roughly equal slice of compute even when one team submits 10x more jobs than the others.

We simulate a small case: 4 GPUs, three teams (A/B/C) with very different submission patterns, and compare two policies:

- **FIFO** — strict arrival order. Whoever queues first gets the GPU.
- **Fair-share (round-robin)** — the scheduler picks the next job from the team with the *least* GPU-hours used so far.

We measure two outcomes: total wall-clock makespan and per-team mean latency (queue time + run time).

In [ ]:
def simulate_scheduler(jobs, n_gpus, policy='fifo'):
    """`jobs` is a list of (team, arrival_time, duration). Returns per-job results.

    Policy 'fifo': always dispatch the earliest-arrived waiting job.
    Policy 'fair': among waiting jobs, dispatch from the team with least cumulative GPU-time.
    """
    gpus_free_at = np.zeros(n_gpus)
    team_used = defaultdict(float)
    results = []
    remaining = sorted(enumerate(jobs), key=lambda x: x[1][1])  # by arrival time
    pending = []  # list of (orig_idx, team, arrival, duration)
    while remaining or pending:
        next_gpu = int(np.argmin(gpus_free_at))
        ready_time = gpus_free_at[next_gpu]
        # Move jobs that have arrived by `ready_time` into pending.
        while remaining and remaining[0][1][1] <= max(ready_time, remaining[0][1][1]):
            idx, (team, arr, dur) = remaining.pop(0)
            pending.append((idx, team, arr, dur))
        if not pending:
            # No one waiting; jump time forward to the next arrival.
            idx, (team, arr, dur) = remaining.pop(0)
            pending.append((idx, team, arr, dur))
        if policy == 'fifo':
            pending.sort(key=lambda x: x[2])
            choice = pending.pop(0)
        elif policy == 'fair':
            # Pick the team with the least cumulative usage, breaking ties by arrival.
            pending.sort(key=lambda x: (team_used[x[1]], x[2]))
            choice = pending.pop(0)
        else:
            raise ValueError(policy)
        idx, team, arr, dur = choice
        start = max(ready_time, arr)
        finish = start + dur
        gpus_free_at[next_gpu] = finish
        team_used[team] += dur
        results.append({'idx': idx, 'team': team, 'arrival': arr,
                        'start': start, 'finish': finish, 'wait': start - arr})
    return results

# Workload: team A submits a burst of 20 short jobs at t=0; teams B and C each
# submit 5 jobs trickled in over the first 10 time units.
rng_sched = np.random.default_rng(11)
jobs = []
for _ in range(20):
    jobs.append(('A', 0.0, float(rng_sched.uniform(2, 5))))
for _ in range(5):
    jobs.append(('B', float(rng_sched.uniform(0, 10)), float(rng_sched.uniform(2, 5))))
for _ in range(5):
    jobs.append(('C', float(rng_sched.uniform(0, 10)), float(rng_sched.uniform(2, 5))))

fifo_results = simulate_scheduler(jobs, n_gpus=4, policy='fifo')
fair_results = simulate_scheduler(jobs, n_gpus=4, policy='fair')

def summarise(results, label):
    waits_by_team = defaultdict(list)
    for r in results:
        waits_by_team[r['team']].append(r['wait'])
    makespan = max(r['finish'] for r in results)
    print(f'\n{label:6s} makespan = {makespan:.1f}')
    for team in ['A', 'B', 'C']:
        mean_wait = float(np.mean(waits_by_team[team]))
        print(f'  team {team}: mean queue wait = {mean_wait:5.2f}  ({len(waits_by_team[team])} jobs)')
    return makespan, {t: float(np.mean(w)) for t, w in waits_by_team.items()}

fifo_make, fifo_waits = summarise(fifo_results, 'FIFO')
fair_make, fair_waits = summarise(fair_results, 'FAIR')

teams = ['A', 'B', 'C']
x = np.arange(len(teams))
w = 0.35
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.bar(x - w / 2, [fifo_waits[t] for t in teams], w, color=ROSE, label=f'FIFO  (makespan {fifo_make:.1f})')
ax.bar(x + w / 2, [fair_waits[t] for t in teams], w, color=TEAL, label=f'fair-share  (makespan {fair_make:.1f})')
ax.set_xticks(x)
ax.set_xticklabels([f'team {t}' for t in teams])
ax.set_ylabel('mean queue wait')
ax.set_title('Fair-share trims tail wait for late-arriving teams at near-zero makespan cost')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

FIFO punishes the late-arriving teams: team A submitted its burst at $t=0$ and locked the whole queue, forcing B and C to wait behind 20 jobs even though they have legitimate work too. The fair-share policy interleaves the queue so each team gets roughly its share of the GPUs, and the makespan stays almost identical because the *total* work hasn't changed — only the order.

Real-world fair-share systems (Slurm, Kueue, YARN Capacity Scheduler) do considerably more than this: weighted shares, hierarchical queues, preemption, and gang scheduling for distributed jobs. The principle they all start from is the one here — dispatch from the team with the smallest cumulative usage.

## ✏️ Your turn — implement an M/M/1 wait-time check

Your goal: implement `mm1_wait(rho, mu)` that returns the closed-form expected queue wait time for an M/M/1 queue:

$$W_q = \frac{\rho}{\mu \, (1 - \rho)}$$

Then assert two boundary behaviours: at $\rho = 0.5, \mu = 1$ the wait is exactly $1.0$; at $\rho = 0.9, \mu = 1$ the wait is exactly $9.0$ (the 'one extra 10% of utilisation costs 8 extra service-times' result).

In [ ]:
def mm1_wait(rho, mu):
    """Expected queue wait time for an M/M/1 queue.

    TODO(you):
    1. Validate 0 <= rho < 1 (return float('inf') if rho >= 1, raise if rho < 0).
    2. Return rho / (mu * (1 - rho)).
    """
    # TODO
    return ...

In [ ]:
w_half = mm1_wait(0.5, 1.0)
w_ninety = mm1_wait(0.9, 1.0)
w_saturated = mm1_wait(1.0, 1.0)

print(f'Wq(rho=0.5) = {w_half}    (expected 1.0)')
print(f'Wq(rho=0.9) = {w_ninety}    (expected 9.0)')
print(f'Wq(rho=1.0) = {w_saturated}  (expected inf)')

assert abs(w_half - 1.0) < 1e-9, f'Wq(0.5, 1) should be 1.0, got {w_half}'
assert abs(w_ninety - 9.0) < 1e-9, f'Wq(0.9, 1) should be 9.0, got {w_ninety}'
assert w_saturated == float('inf'), f'Wq(1.0, 1) should be inf, got {w_saturated}'
print('\n✅ mm1_wait matches the closed-form expectations across the regime.')

<details>
<summary>Solution</summary>

```python
def mm1_wait(rho, mu):
    if rho < 0:
        raise ValueError('rho must be non-negative')
    if rho >= 1:
        return float('inf')
    return rho / (mu * (1 - rho))
```

Two operational reasons this formula is worth memorising:

- *The constant in front of $1/(1-\rho)$ is `rho / mu`, not `1/mu`.* The dependence on $\mu$ matters: doubling service rate halves the wait at any utilisation, not just shifts it.
- *Real clusters are M/M/c, not M/M/1.* With $c$ servers the Erlang-C formula governs wait time — the divergence as $\rho \to 1$ is the same shape, but the coefficient drops with $c$. That's why 'more cluster' helps more than 'faster cluster' for high-arrival workloads.

</details>

## Recap

- A **DAG executor with retries** turns flaky pipelines into reliable ones at zero engineering cost beyond making each step idempotent. Three retries on a 60%-failure step is a >98% step.
- **Queueing theory** quantifies why 95% utilisation feels broken: $W_q = \rho / (\mu (1 - \rho))$ blows up near $\rho = 1$. Aim for 70–80% steady-state.
- **Fair-share scheduling** trims tail latency for late-arriving teams without sacrificing makespan, by dispatching from the team with the least cumulative usage.
- Closed-form formulas like $W_q$ are how you reason about platform behaviour *before* the cluster melts down. Memorising the shape is worth more than memorising the exact constant.